In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))

from pipeline.bronze.cdc import to_bronze

In [0]:
dbutils.widgets.text("topic", "")
dbutils.widgets.text("base", "s3://motorsport-data-lake")

TOPIC = dbutils.widgets.get("topic")
BASE = dbutils.widgets.get("base")

In [0]:
if not TOPIC:
    raise ValueError("topic parameter is required, e.g. motorsport.public.tracks")

NAME = TOPIC.split(".")[-1]
TARGET = f"motorsport.bronze.cdc_{NAME}"

In [0]:
kafka_bootstrap = dbutils.secrets.get("motorsport", "kafka_bootstrap")
api_key = dbutils.secrets.get("motorsport", "kafka_api_key")
api_secret = dbutils.secrets.get("motorsport", "kafka_api_secret")

jaas = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
    f'required username="{api_key}" password="{api_secret}";'
)

In [0]:
raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .option("kafka.group.id", f"motorsport-databricks-bronze-cdc-{NAME}")
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
query = (
    to_bronze(raw)
    .writeStream
    .option("checkpointLocation", f"{BASE}/checkpoints/bronze_cdc_{NAME}")
    .trigger(availableNow=True)
    .toTable(TARGET)
)

In [0]:
query.awaitTermination()

In [0]:
dbutils.jobs.taskValues.set(key="table", value=NAME)